In [0]:
1+1

## scAuditor_agent — Project Overview

**Bundle:** `scAuditor_agent` — a Databricks App bundle (deploy AFTER `scAuditor_infra`)

**Purpose:** AI-powered screen capture audit tool for regulated industries. An auditor uses natural language to direct an agent that navigates web systems via Playwright, captures screenshots, extracts data with `ai_parse_document`, records findings, and generates audit reports.

---

### Architecture

```
Browser (React 19 + Tailwind + Vite) → Express (AppKit) → Agent Loop (LLM tool-use, max 15 rounds)
  → Playwright (headless Chromium) → target system screenshots
  → ai_parse_document (SQL warehouse, OBO) → extraction results
  → Lakebase (Postgres 17) → operational state, patterns, memory
  → UC Volumes → screenshots and report files
  → Foundation Model API (databricks-claude-sonnet-4 via ai_query) → intelligence
```

### Stack

| Layer | Technology |
| --- | --- |
| Frontend | React 19, react-router 7, Tailwind CSS 4, lucide-react, Databricks AppKit UI |
| Backend | Node.js + Express via `@databricks/appkit` 0.20.3 |
| Browser automation | Playwright 1.52 (headless Chromium) |
| LLM | `databricks-claude-sonnet-4` via `ai_query()` through Analytics plugin |
| OLTP database | Lakebase Autoscaling (Postgres 17) |
| File storage | UC Volumes (screenshots + reports) |
| Observability | OpenTelemetry SDK → UC Delta tables (logs, traces, metrics) |
| Build tools | tsdown (server), Vite / rolldown-vite (client), TypeScript 5.9 |

---

### Bundle Targets

| Target | Workspace | Catalog | Schema | Mode |
| --- | --- | --- | --- | --- |
| `dev` (default) | fevm-hls-fde | `hls_fde_dev` | `dev_matthew_giglia_sc_auditor` | development |
| `hls_fde` | fevm-hls-fde | `hls_fde` | `sc_auditor` | production |
| `prod` | fevm-hls-fde | (TBD) | (TBD) | production (placeholder) |

---

### Agent — 15 Tools in 8 Categories

| Category | Tools | Description |
| --- | --- | --- |
| Browser (7) | `navigate_to_url`, `click_element`, `type_text`, `press_key`, `take_screenshot`, `wait_for_element`, `get_page_content` | Playwright headless Chromium control |
| Extraction (1) | `extract_from_screenshot` | `ai_parse_document` via SQL warehouse |
| Findings (1) | `record_finding` | Store findings in UC with severity/category/evidence |
| Memory (2) | `recall_memory`, `store_memory` | Long-term knowledge per-user, optionally per-system |
| Patterns (2) | `recall_pattern`, `save_pattern` | Navigation pattern learning and reuse |
| Auth (1) | `login_to_system` | Authenticate via secret scope or UC connection |
| Reports (1) | `generate_report` | HTML report with LLM executive summary |
| Workflow (3) | `start_workflow`, `update_workflow_step`, `complete_workflow` | Step-by-step progress tracking with pattern stats |

---

### Lakebase Tables (Operational — Postgres)

| Table | Purpose |
| --- | --- |
| `app.agent_sessions` | Session lifecycle state |
| `app.agent_messages` | Full conversation history per session |
| `app.navigation_patterns` | Curated/learned navigation patterns per target system |
| `app.agent_memory` | Long-term agent knowledge (preferences, facts, system quirks) |
| `app.active_audit_workflows` | In-progress workflow step tracking |
| `app.credential_references` | Pointers to credentials (never stores actual secrets) |

### UC Analytical Tables (Delta — from scAuditor_infra)

`audit_sessions`, `audit_screenshots`, `audit_extractions`, `audit_findings`, `audit_reports`, `target_systems`

---

### Frontend Pages

| Route | Page | Description |
| --- | --- | --- |
| `/` | Dashboard | Stats cards + recent sessions table |
| `/audit`, `/audit/:sessionId` | Audit | Split-pane workspace: screenshots (60%) + agent chat (40%) |
| `/history` | History | Searchable, filterable past sessions |
| `/patterns` | Patterns | Visual pattern editor with step timeline |
| `/settings` | Settings | Credential management (user + admin M2M) + preferences |

---

### AppKit Plugins

| Plugin | Purpose |
| --- | --- |
| `analytics` | SQL warehouse access for `ai_parse_document`, `ai_query`, analytical queries |
| `lakebase` | Postgres connection pool for OLTP tables |
| `files` | UC Volume file upload/download for screenshots and reports |
| `server` | Express HTTP server with middleware |

### OTel Telemetry Tables (dev target)

| Table |
| --- |
| `hls_fde_dev.dev_matthew_giglia_sc_auditor.sc_auditor_agent_otel_logs` |
| `hls_fde_dev.dev_matthew_giglia_sc_auditor.sc_auditor_agent_otel_traces` |
| `hls_fde_dev.dev_matthew_giglia_sc_auditor.sc_auditor_agent_otel_metrics` |

---

### Session History (5 sessions)

| Date | Summary |
| --- | --- |
| 2026-04-25 | Nav gradient, footer enhancement, body surfaces, HC dark mode fix |
| 2026-04-24 | Page refresh — brand CDN icons, favicon, 120px navbar hero, 10 deploys |
| 2026-04-24 | Frontend brand refresh — semantic tokens, ThemeProvider, shared components |
| 2026-04-24 | Deploy fixes (TS build, Lakebase init), OTel observability dashboard |
| 2026-04-23 | App bundle scaffold, agent core, UI, and intelligence features |

## Key File Map

```
scAuditor_agent/
├── databricks.yml                       # Bundle config: 3 targets, variables, includes
├── README.md                            # Comprehensive project documentation
├── resources/
│   └── sc_auditor.app.yml               # App resource: secrets, Lakebase, OTel, permissions
├── sc-auditor-app/                      # Databricks App (AppKit)
│   ├── app.yaml                         # Runtime env injection (valueFrom secrets + postgres)
│   ├── appkit.plugins.json              # Plugin config: analytics, lakebase, files, server
│   ├── package.json                     # Node.js deps (AppKit 0.20.3, Playwright, React 19, OTel)
│   ├── client/
│   │   └── src/
│   │       ├── App.tsx                  # Router: 5 pages with Layout (Navbar + Footer)
│   │       ├── ThemeProvider.tsx         # Dark/light mode toggle
│   │       ├── components/              # Reusable UI: Card, Badge, Button, Input, etc.
│   │       ├── lib/                     # API client, brand assets
│   │       └── pages/                   # dashboard, audit, history, patterns, settings
│   └── server/
│       ├── server.ts                    # AppKit entry: plugin init, schema bootstrap, routes
│       ├── otel.ts                      # OpenTelemetry Node SDK auto-instrumentation
│       ├── db/init-schema.ts             # Lakebase schema bootstrap (6 OLTP tables)
│       ├── agent/
│       │   ├── auditor-agent.ts          # Core LLM tool-use loop (max 15 rounds)
│       │   ├── types.ts                  # ToolDefinition, AgentMessage, AgentContext
│       │   ├── prompts/system-prompt.ts  # Agent system prompt with workflow guidelines
│       │   └── tools/                    # 15 tools across 8 categories
│       │       ├── index.ts              # Tool registry aggregator
│       │       ├── browser-tools.ts      # 7 Playwright browser control tools
│       │       ├── extraction-tools.ts   # ai_parse_document extraction
│       │       ├── finding-tools.ts      # Record findings to UC
│       │       ├── memory-tools.ts       # Long-term agent memory (Lakebase)
│       │       ├── pattern-tools.ts      # Navigation pattern learning
│       │       ├── login-tools.ts        # Target system authentication
│       │       ├── report-tools.ts       # HTML report generation
│       │       └── workflow-tools.ts     # Step-by-step workflow tracking
│       ├── plugins/browser-agent/
│       │   └── browser-controller.ts     # Playwright headless Chromium wrapper
│       ├── routes/                      # 6 Express route modules
│       └── services/report-generator.ts  # HTML report builder with LLM summaries
├── fixtures/
│   ├── sessions/                        # 5 session summary files + INDEX.md
│   └── Genie Code Session Starter       # This notebook
└── shared/types.ts                      # Shared TypeScript types (client + server)
```

## Companion Bundle — scAuditor_infra

`scAuditor_infra` is the infrastructure-only DAB that must be deployed **first**. It provisions:

| Resource | Description |
| --- | --- |
| UC Schema | `{catalog}.sc_auditor` (or dev-prefixed) |
| Volumes | `screenshots` (PNGs), `reports` (generated files) |
| SQL Warehouse | 2X-Small serverless PRO |
| Lakebase | PG17 autoscaling (0.5–4 CU) |
| Secret Scope | `sc_auditor_credentials` (SPN creds, workspace URL) |
| Bootstrap Job | Two-task: SPN creation → DDL + grants |
| Analytical Tables | `audit_sessions`, `audit_screenshots`, `audit_extractions`, `audit_findings`, `audit_reports`, `target_systems` (Delta, CLUSTER BY AUTO, CDF, VARIANT columns) |

**Cross-bundle convention:** This agent bundle references infra objects by value (`${var.catalog}`, `${var.schema}`, `${var.secret_scope_name}`) since DAB cross-bundle substitutions are not supported. Values must be kept in sync.

## OTel Telemetry Tables (dev target)

The app has OpenTelemetry enabled via `telemetry_export_destinations` in the app resource YAML. The platform injects an OTLP collector at `localhost:4314` and streams real-time data to these UC Delta tables:

| Table | FQN (dev target) |
| --- | --- |
| Logs | `hls_fde_dev.dev_matthew_giglia_sc_auditor.sc_auditor_agent_otel_logs` |
| Traces | `hls_fde_dev.dev_matthew_giglia_sc_auditor.sc_auditor_agent_otel_traces` |
| Metrics | `hls_fde_dev.dev_matthew_giglia_sc_auditor.sc_auditor_agent_otel_metrics` |

Custom instrumentation (`server/otel.ts`) adds Express/HTTP auto-tracing. Trace sampling is set to 10% (`OTEL_TRACES_SAMPLER_ARG: 0.1`).

In [0]:
%sql
-- OTel Telemetry Overview — Logs
-- Volume by date, severity distribution, and top log sources
WITH log_stats AS (
  SELECT
    date,
    severity_text,
    attributes:`app.log_source`::STRING AS log_source,
    attributes:`app.deployment_id`::STRING AS deployment_id,
    COUNT(*) AS cnt
  FROM hls_fde_dev.dev_matthew_giglia_sc_auditor.sc_auditor_agent_otel_logs
  WHERE date >= current_date() - INTERVAL 7 DAYS
  GROUP BY ALL
)
SELECT
  date,
  severity_text,
  log_source,
  deployment_id,
  cnt
FROM log_stats
ORDER BY date DESC, cnt DESC

In [0]:
%sql
-- OTel Traces — Span summary by instrumentation library and operation
SELECT
  date,
  instrumentation_scope.name AS instrumentation_lib,
  instrumentation_scope.version AS lib_version,
  name AS span_name,
  kind,
  status.code AS status_code,
  COUNT(*) AS span_count,
  ROUND(AVG((end_time_unix_nano - start_time_unix_nano) / 1e6), 2) AS avg_duration_ms,
  ROUND(MAX((end_time_unix_nano - start_time_unix_nano) / 1e6), 2) AS max_duration_ms,
  ROUND(MIN((end_time_unix_nano - start_time_unix_nano) / 1e6), 2) AS min_duration_ms
FROM hls_fde_dev.dev_matthew_giglia_sc_auditor.sc_auditor_agent_otel_traces
WHERE date >= current_date() - INTERVAL 7 DAYS
GROUP BY ALL
ORDER BY span_count DESC
LIMIT 50

In [0]:
%sql
-- OTel Metrics — Available metrics and their types
SELECT
  date,
  name AS metric_name,
  description,
  metric_type,
  instrumentation_scope.name AS instrumentation_lib,
  COUNT(*) AS data_points
FROM hls_fde_dev.dev_matthew_giglia_sc_auditor.sc_auditor_agent_otel_metrics
WHERE date >= current_date() - INTERVAL 7 DAYS
GROUP BY ALL
ORDER BY data_points DESC

In [0]:
%sql
-- OTel Traces — HTTP request latency distribution (server-side spans)
SELECT
  date,
  attributes:`http.method`::STRING AS http_method,
  attributes:`http.route`::STRING AS http_route,
  attributes:`http.target`::STRING AS http_target,
  attributes:`http.status_code`::STRING AS status_code,
  COUNT(*) AS request_count,
  ROUND(AVG((end_time_unix_nano - start_time_unix_nano) / 1e6), 2) AS avg_latency_ms,
  ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY (end_time_unix_nano - start_time_unix_nano) / 1e6), 2) AS p50_ms,
  ROUND(PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY (end_time_unix_nano - start_time_unix_nano) / 1e6), 2) AS p95_ms,
  ROUND(PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY (end_time_unix_nano - start_time_unix_nano) / 1e6), 2) AS p99_ms
FROM hls_fde_dev.dev_matthew_giglia_sc_auditor.sc_auditor_agent_otel_traces
WHERE date >= current_date() - INTERVAL 7 DAYS
  AND kind = 'SPAN_KIND_SERVER'
GROUP BY ALL
ORDER BY request_count DESC
LIMIT 30

In [0]:
%sql
-- OTel Traces — Error spans (non-OK status)
SELECT
  time,
  name AS span_name,
  kind,
  status.code AS status_code,
  status.message AS status_message,
  attributes:`http.method`::STRING AS http_method,
  attributes:`http.target`::STRING AS http_target,
  attributes:`http.status_code`::STRING AS http_status,
  ROUND((end_time_unix_nano - start_time_unix_nano) / 1e6, 2) AS duration_ms,
  instrumentation_scope.name AS instrumentation_lib
FROM hls_fde_dev.dev_matthew_giglia_sc_auditor.sc_auditor_agent_otel_traces
WHERE date >= current_date() - INTERVAL 7 DAYS
  AND status.code = 'STATUS_CODE_ERROR'
ORDER BY time DESC
LIMIT 50

In [0]:
%sql
-- Investigate missing SPAN_KIND_SERVER traces
-- Metrics are UNSAMPLED (always collected); traces are SAMPLED at 10%
-- Compare actual request volume (metrics) vs sampled trace count

WITH server_request_volume AS (
  -- Aggregate histogram counts from http.server.duration metrics
  SELECT
    date,
    histogram.attributes:`http.method`::STRING AS http_method,
    histogram.attributes:`http.status_code`::STRING AS status_code,
    SUM(histogram.count) AS total_requests,
    ROUND(SUM(histogram.sum) / NULLIF(SUM(histogram.count), 0), 2) AS avg_duration_ms
  FROM hls_fde_dev.dev_matthew_giglia_sc_auditor.sc_auditor_agent_otel_metrics
  WHERE date >= current_date() - INTERVAL 7 DAYS
    AND name = 'http.server.duration'
    AND metric_type = 'histogram'
  GROUP BY date, http_method, status_code
),
server_trace_count AS (
  SELECT
    date,
    COUNT(*) AS sampled_server_spans
  FROM hls_fde_dev.dev_matthew_giglia_sc_auditor.sc_auditor_agent_otel_traces
  WHERE date >= current_date() - INTERVAL 7 DAYS
    AND kind = 'SPAN_KIND_SERVER'
  GROUP BY date
)
SELECT
  v.date,
  v.http_method,
  v.status_code,
  v.total_requests AS actual_requests_unsampled,
  v.avg_duration_ms,
  COALESCE(t.sampled_server_spans, 0) AS sampled_server_traces,
  ROUND(v.total_requests * 0.10) AS expected_sampled_at_10pct,
  '10% sampling too aggressive for dev traffic' AS diagnosis
FROM server_request_volume v
LEFT JOIN server_trace_count t ON v.date = t.date
ORDER BY v.date DESC, v.total_requests DESC

## Diagnosis: Missing `SPAN_KIND_SERVER` Traces

### Evidence

| Signal | Source | Apr 24 | Apr 25 | Conclusion |
| --- | --- | --- | --- | --- |
| HTTP server metrics (`http.server.duration`) | `@opentelemetry/instrumentation-http` | 124,345 requests | 20,314 requests | HTTP instrumentation IS running |
| HTTP server traces (`SPAN_KIND_SERVER`) | `@opentelemetry/instrumentation-http` | **0 spans** | **0 spans** | Traces NOT being created or exported |
| HTTP client traces (`SPAN_KIND_CLIENT`) | `@opentelemetry/instrumentation-http` | 30 spans | 0 | Client-side trace pipeline works |
| Lakebase/PG traces | `@databricks/lakebase`, `instrumentation-pg` | 394 spans | 27 | Non-HTTP trace pipeline works |

At 10% sampling with 124K requests, we'd expect \~12,400 sampled server spans. Getting **zero** proves this is not a sampling issue.

### Root Cause Analysis

The OTel SDK creates metrics and traces via different code paths in `@opentelemetry/instrumentation-http`. Metrics use synchronous histogram observations on every request. Traces require the instrumentation's request/response hooks to fire and create span objects that flow through the sampling pipeline to the `OTLPTraceExporter`.

**Two issues compound:**

#### 1. Protocol Mismatch (`-proto` exporters vs gRPC sidecar)

The platform injects `OTEL_EXPORTER_OTLP_PROTOCOL=grpc` and runs a **gRPC** collector sidecar at `localhost:4314`. But `otel.ts` explicitly creates HTTP/protobuf exporters:

```ts
import { OTLPTraceExporter } from '@opentelemetry/exporter-trace-otlp-proto';  // HTTP/protobuf
import { OTLPMetricExporter } from '@opentelemetry/exporter-metrics-otlp-proto'; // HTTP/protobuf
import { OTLPLogExporter } from '@opentelemetry/exporter-logs-otlp-proto';      // HTTP/protobuf
```

The `-proto` package sends HTTP POST to `http://localhost:4314/v1/traces`. The sidecar accepts this for most payloads, but server spans may be silently dropped due to content negotiation or size limits. The gRPC exporter (`-grpc` package) would use the native protocol the sidecar expects.

#### 2. ESM Hook Registration for Server-Side Instrumentation

The app uses ESM (`"type": "module"`). OTel's HTTP instrumentation patches `http.createServer()` via `import-in-the-middle` ESM hooks (registered by `NodeSDK` when loaded via `--import`). While client-side `http.request()` patching works (proven by CLIENT spans), the server-side `createServer()` hook may not fire correctly in this ESM + AppKit combination.

AppKit's `server` plugin creates the Express server internally. If it imports `http` from `node:http` before the ESM hook fully registers, server spans won't be generated — but metric observations (which use a different API path) still work.

### Recommended Fix

Switch to gRPC exporters and increase sampling for dev:

**`otel.ts` changes:**
```ts
// BEFORE (HTTP/protobuf — protocol mismatch with platform sidecar)
import { OTLPTraceExporter } from '@opentelemetry/exporter-trace-otlp-proto';
import { OTLPMetricExporter } from '@opentelemetry/exporter-metrics-otlp-proto';
import { OTLPLogExporter } from '@opentelemetry/exporter-logs-otlp-proto';

// AFTER (gRPC — matches platform's OTEL_EXPORTER_OTLP_PROTOCOL=grpc)
import { OTLPTraceExporter } from '@opentelemetry/exporter-trace-otlp-grpc';
import { OTLPMetricExporter } from '@opentelemetry/exporter-metrics-otlp-grpc';
import { OTLPLogExporter } from '@opentelemetry/exporter-logs-otlp-grpc';
```

**`package.json` changes:**
```diff
- "@opentelemetry/exporter-logs-otlp-proto": "^0.203.0",
- "@opentelemetry/exporter-metrics-otlp-proto": "^0.203.0",
- "@opentelemetry/exporter-trace-otlp-proto": "^0.203.0",
+ "@opentelemetry/exporter-logs-otlp-grpc": "^0.203.0",
+ "@opentelemetry/exporter-metrics-otlp-grpc": "^0.203.0",
+ "@opentelemetry/exporter-trace-otlp-grpc": "^0.203.0",
+ "@grpc/grpc-js": "^1.12.0",
```

**`app.yaml` — bump sampling for dev:**
```yaml
  - name: OTEL_TRACES_SAMPLER_ARG
    value: '1.0'   # 100% during dev; revert to 0.1 for production
```

If server spans still don't appear after the gRPC switch, the ESM hook issue is the primary cause. Workaround: add manual Express middleware that creates server spans explicitly.